In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error,r2_score)


In [2]:
df = pd.read_csv("D:\\BOOKS FOR TEACHING\\MLOPS_2026_Upgrade\\mlops_learn\\data\\data.csv")

df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [3]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=56)

First, we configure MLflow to store all experiment tracking metadata inside a local SQLite database named mlflow.db located in the current working directory

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

Then we create an experiment named **Advertising Sales Prediction**

In [5]:
mlflow.set_experiment("Advertising Sales Prediction")

2026/08/27 14:46:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/27 14:46:58 INFO mlflow.store.db.utils: Updating database tables
2026/08/27 14:47:01 INFO mlflow.tracking.fluent: Experiment with name 'Advertising Sales Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/BOOKS FOR TEACHING/MLOPS_2026_Upgrade/mlops_learn/notebooks/mlruns/1', creation_time=1787822221268, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787822221268, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

Now, we create a **run** inside the experiment **Advertising Sales Prediction**

In [6]:
with mlflow.start_run(run_name="Linear Regression"):

    model = LinearRegression()
    
    model.fit(xtrain, ytrain)

    y_pred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

**Let's open the mlflow User Interface**

Open a terminal in the environment where MLflow is installed and run:

```bash 
mlflow ui
```

**Now, we create another run using a Ridge Regression model**

In [7]:
with mlflow.start_run(run_name="Ridge Regression"):

    model = Ridge(alpha=1.0)
    model.fit(xtrain, ytrain)

    ypred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Log the metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    # Log Artifacts
    mlflow.sklearn.log_model(sk_model = model, name="Ridge_Reg_Model")



**Artifacts are the concrete output files generated by a run.**

Now, we use the **autologging** feature in Mlflow.

Autologging allows MLflow to automatically cpature much of the information.

In [8]:
# Turn on Scikit-learn autologging.
mlflow.sklearn.autolog()

In [13]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model.fit(xtrain, ytrain)

    test_pred = model.predict(xtest)

    test_rmse = root_mean_squared_error(ytest, test_pred) 
    test_mae = mean_absolute_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics
    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    })


2026/08/27 15:07:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


### Registering the Model

The MLflow Model Registry is a central hub (like an app store or catalog) to keep all production-ready models in one shared, searchable place instead of scattered across folders or runs.

Among all the experiments you perform, register the final selected model.

In a production environment, the models are **continuously trained**. This means the registered models would have many versions:

```text
Advertising_Sales_Model
│
├── Version 1
├── Version 2
├── Version 3   
└── Version 4  
```



In [14]:
# For registering the model, we require the model URI.
# URI is a unique identifier for a model.

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/eb17b1863f2b4cc88ac05a34e547d427/model


Now let's register the model

In [15]:
registered_model = mlflow.register_model(
     model_uri=model_uri,
     name="Advertising_Sales_Model"
 )

registered_model

Successfully registered model 'Advertising_Sales_Model'.
2026/08/27 15:08:15 WARNING mlflow.tracking._model_registry.fluent: Run with id eb17b1863f2b4cc88ac05a34e547d427 has no artifacts at artifact path 'model', registering model based on models:/m-685981c983a34c1f8e7445143e530cc7 instead
Created version '1' of model 'Advertising_Sales_Model'.


<ModelVersion: aliases=[], creation_timestamp=1787823495927, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1787823495927, metrics=None, model_id=None, name='Advertising_Sales_Model', params=None, run_id='eb17b1863f2b4cc88ac05a34e547d427', run_link=None, source='models:/m-685981c983a34c1f8e7445143e530cc7', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

### Model Aliases

Model Aliasing gives a nickname (like "current_best" or "production") to a specific version of the registered model.

Instead of typing exact numbers like Version 1, Version 2, or Version 15, you just use the nickname.

```text
Advertising_Sales_Model
│
├── Version 1
├── Version 2
├── Version 3   ← champion
└── Version 4   ← challenger
```

`champion` means:

> the currently preferred model.

`challenger` means:

> a new candidate being evaluated as a possible replacement.

In [17]:
from mlflow import MlflowClient

client = MlflowClient()

# 1. Assign an alias to a specific version
# (Sets the alias 'champion' to Version 2 of 'fraud_detector')
client.set_registered_model_alias(
    name="Advertising_Sales_Model", 
    alias="champion", 
    version="1"
)

### Loading a registered model

In [18]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion" 
)

# Generate new predictions
new_data = pd.DataFrame({
    "TV": [150.0],
    "Radio": [25.0],
    "Newspaper": [30.0]
})

print(model.predict(new_data))

[14.2199203]
